# Call Labeling Benchmark: keyword rules vs supervised ML vs frontier LLMs vs System One models

**Question:** a support team wants every conversation labeled: why the customer reached out, what the agent did, how far it went. Which labeler gets it right, and what does each one cost in time and money at scale?

**Contenders**

| Family | Models | Needs labeled data? |
|---|---|---|
| Keyword rules | one regex per label | no |
| Supervised ML | TF-IDF + logistic regression, with a learning curve | yes (up to ~8,000 conversations) |
| Frontier LLMs | Claude Sonnet 5, GPT-6 Sol, Gemini 3.8 Flash, Claude Haiku 4.5 (cheap tier) | no |
| System One | Jev 1.13 (TypeSafe, API), Laya (Convai, open weights, runs locally) | no |

**The rules every model plays by**

| Rule | Why |
|---|---|
| One conversation per call, all questions at once | How labeling runs in production. Jev and Laya take several typed questions natively; the LLMs return the same answers as one JSON object. |
| Identical question wording and option descriptions for everyone | Built once from a single `QUESTIONS` table, then rendered for each API. Nothing is tuned per model. |
| Only agent and customer lines are shown | The system action log is the ground truth; it is never in the transcript. |
| Structured outputs for every LLM | The "LLMs return broken JSON" argument is tested, not assumed: any invalid answer is logged and scored as wrong. |
| Provider-default reasoning, no tools | "Out of the box", same as the attrition run. |
| Questions and descriptions frozen before the test split is scored | The dev split is used only to prove the wiring. |
| Test set: all 1,000 usable ABCD test conversations | Big enough that confidence intervals separate models (the attrition run had 16 positives). 4 of the 1,004 carry an intent missing from ABCD's own ontology, so no option would be right. |

**The questions** use all three System One answer types: two `choice` questions (reason for contact, 10 options; specific intent, 55 options), four yes/no `noul` questions (identity verified, refund offered, promo code given, handed to another team), and one ordinal `score` (how far the agent went: information only, changed something, gave money back). Every answer has ground truth in the dataset, logged by the system rather than judged by an annotator.

**Sections:** 1 Load, 2 Questions and ground truth, 3 Scorers, 4 Keyword baseline, 5 Supervised baseline and learning curve, 6 Prompt and requests, 7 Models, 8 Smoke test, 9 Run, 10 Results, 11 Memorization probe, 12 Cost at scale, Caveats.

## 1. Load

Same pinned download as `01_explore.ipynb`, cached outside the repo.

In [ ]:
%pip install -q pandas numpy scikit-learn matplotlib tiktoken typesafe-sdk anthropic openai google-genai

import gzip
import json
import os
import random
import re
import time
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ABCD_COMMIT = "6b8700ce67c6b37b062dd7a60abc76d7ef832a97"   # asappresearch/abcd, pinned
ABCD_FILES = ["data/abcd_v1.1.json.gz", "data/kb.json", "data/ontology.json"]
CACHE_DIR = Path.home() / ".cache" / "abcd" / ABCD_COMMIT
for file in ABCD_FILES:
    local_path = CACHE_DIR / file
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/asappresearch/abcd/{ABCD_COMMIT}/{file}", local_path)

with gzip.open(CACHE_DIR / "data/abcd_v1.1.json.gz") as f:
    abcd = json.load(f)
policy_actions = json.loads((CACHE_DIR / "data/kb.json").read_text())   # the 55 intents that have an action policy
print({split: len(conversations) for split, conversations in abcd.items()})

## 2. Questions and ground truth

`QUESTIONS` is the single source of truth: the wording and option descriptions every model sees. Option descriptions were written from the label names and the dataset's own ontology, before any model was run.

- **reason** (choice, 10): the scenario's `flow`.
- **intent** (choice, 55): the dataset's official 55 intents. For the two FAQ flows the scenario's `subflow` is a specific FAQ article (`boots_how_2`, `timing_1`); it maps to its intent by the name before the article number (`boots`, `timing`), as in the ABCD ontology.
- **four yes/no questions**: whether the matching action appears in the system log.
- **resolution** (score, 3 ordered levels) from the log: money back (refund or promo code) > changed something (order, account, password, purchase) > information only.

In [ ]:
FLOW_DESCRIPTIONS = {
    "account_access": "Can't get into their account: forgot username or password, or two-factor authentication problems",
    "manage_account": "Wants to change account details or services: address, name, phone, payment method, added or removed services",
    "order_issue": "Problem with an existing order: unexpected fee, delivery time, payment method, quantity, or wants to change or cancel the order",
    "product_defect": "Received a faulty or wrong item and wants a return or refund, or asks about a refund",
    "purchase_dispute": "Disputes a charge or price: price dropped, competitor is cheaper, item out of stock, promo code failed, billed for something not bought or already returned",
    "shipping_issue": "Shipping problem: package status, change shipping details, missing package, or shipping cost",
    "single_item_query": "General question about one product (how it works, materials, sizing) with no order problem",
    "storewide_query": "General question about store-wide policies: pricing, timing, membership, or return policy",
    "subscription_inquiry": "Question about their subscription or membership: status, due date or amount, paying the bill, or extending it",
    "troubleshoot_site": "The website isn't working: slow, search results wrong, shopping cart or credit card not working",
}
FAQ_FLOWS = {"single_item_query", "storewide_query"}

def readable(name):
    return name.replace("_", " ").replace("2fa", "two-factor authentication")

# Intent options: ABCD's 55 official intents, each described as "<flow>: <intent>" straight from the ontology names
ontology = json.loads((CACHE_DIR / "data/ontology.json").read_text())
INTENT_FLOW = {intent: flow for flow, intents in ontology["intents"]["subflows"].items() for intent in intents}
INTENT_DESCRIPTIONS = {intent: f"{readable(flow)}: {readable(intent)}" for intent, flow in sorted(INTENT_FLOW.items())}

RESOLUTION_LEVELS = {   # ordered, lowest first; the index is the score
    "information_only": "Information only: the agent answered questions, looked things up or passed the issue on, but changed nothing and gave nothing",
    "changed_something": "Changed something: the agent updated an order or account, reset a password, or placed a purchase, but gave no money back",
    "money_back": "Money back: the agent issued a refund or gave the customer a promo code",
}
MONEY_ACTIONS = {"offer-refund", "promo-code"}
CHANGE_ACTIONS = {"update-order", "update-account", "make-password", "make-purchase"}

QUESTIONS = {
    "reason": {"type": "choice", "instructions": "Why did the customer contact the store?", "criteria": FLOW_DESCRIPTIONS},
    "intent": {"type": "choice", "instructions": "What specific request did the customer make?", "criteria": INTENT_DESCRIPTIONS},
    "identity_verified": {"type": "noul", "instructions": "Did the agent verify the customer's identity using their details (for example name, account ID, order ID or zip code)?",
                          "criteria": {"true": "The agent checked the customer's identity", "false": "No identity check was done"}},
    "refund_offered": {"type": "noul", "instructions": "Did the agent issue a refund to the customer?",
                       "criteria": {"true": "A refund was issued", "false": "No refund was issued (refunds may be mentioned, asked about or denied)"}},
    "promo_given": {"type": "noul", "instructions": "Did the agent give the customer a promo code?",
                    "criteria": {"true": "A promo code was given", "false": "No promo code was given"}},
    "handed_off": {"type": "noul", "instructions": "Did the agent pass the issue to another team (a manager, the purchasing department or the website team)?",
                   "criteria": {"true": "The issue was passed to another team", "false": "The agent handled it without passing it on"}},
    "resolution": {"type": "score", "instructions": "How far did the agent go for the customer?", "criteria": RESOLUTION_LEVELS},
}
NOUL_ACTIONS = {"identity_verified": "verify-identity", "refund_offered": "offer-refund",
                "promo_given": "promo-code", "handed_off": "notify-team"}
CHOICE_QUESTIONS = [name for name, question in QUESTIONS.items() if question["type"] == "choice"]
NOUL_QUESTIONS = [name for name, question in QUESTIONS.items() if question["type"] == "noul"]
SCORE_QUESTIONS = [name for name, question in QUESTIONS.items() if question["type"] == "score"]
print(f"{len(INTENT_DESCRIPTIONS)} intent options")

In [ ]:
def conversation_text(conversation):
    # Agent and customer lines only: action lines are the ground truth and never reach a model
    return "\n".join(f"{speaker.capitalize()}: {text}" for speaker, text in conversation["original"] if speaker != "action")

def ground_truth(conversation):
    actions = {turn["targets"][2] for turn in conversation["delexed"] if turn["speaker"] == "action"}
    flow, subflow = conversation["scenario"]["flow"], conversation["scenario"]["subflow"]
    labels = {"reason": flow, "intent": subflow.split("_")[0] if flow in FAQ_FLOWS else subflow}   # FAQ article -> its intent
    labels.update({question: action in actions for question, action in NOUL_ACTIONS.items()})
    labels["resolution"] = 2 if actions & MONEY_ACTIONS else 1 if actions & CHANGE_ACTIONS else 0
    return labels

def build_split(split):
    rows = [{"convo_id": conversation["convo_id"], "text": conversation_text(conversation), **ground_truth(conversation)}
            for conversation in abcd[split]]
    frame = pd.DataFrame(rows).sort_values("convo_id").reset_index(drop=True)   # fixed order for every model
    # 33 conversations (4 in test) carry an intent missing from ABCD's own ontology (`status_questions`, `status_delivery_date`): no right option exists
    off_ontology = ~frame["intent"].isin(INTENT_DESCRIPTIONS)
    print(f"{split}: dropped {off_ontology.sum()} conversation(s) with an off-ontology intent: {sorted(frame.loc[off_ontology, 'intent'])}")
    return frame[~off_ontology].reset_index(drop=True)

train, dev, test = build_split("train"), build_split("dev"), build_split("test")
conversation_by_id = {conversation["convo_id"]: conversation for conversations in abcd.values() for conversation in conversations}

label_summary = {}
for question in NOUL_QUESTIONS:
    label_summary[question] = {"train positives": int(train[question].sum()), "test positives": int(test[question].sum()),
                               "test share": round(test[question].mean(), 3)}
display(pd.DataFrame(label_summary).T)
print("resolution levels in test:", test["resolution"].map(dict(enumerate(RESOLUTION_LEVELS))).value_counts().to_dict())

## 3. Scorers

One scorer per answer type, shared by every model. Bootstrap 95% intervals use the **same resamples for every model** (fixed seed), so intervals are directly comparable.

- **choice**: accuracy (headline) and macro-F1 (every option counts equally).
- **noul**: F2 on "yes" (rewards catching the rare positives, still punishes flagging everything), plus recall, precision and, for models that return a probability, AUC.
- **score**: exact accuracy and quadratic weighted kappa (partial credit for landing one level off).

In [ ]:
from sklearn.metrics import accuracy_score, cohen_kappa_score, f1_score, roc_auc_score

BOOTSTRAP_RESAMPLES = 2000
RESAMPLE_INDEX = np.random.default_rng(SEED).integers(0, len(test), size=(BOOTSTRAP_RESAMPLES, len(test)))

def bootstrap_ci(metric, true_values, predicted_values):
    true_values, predicted_values = np.asarray(true_values), np.asarray(predicted_values)
    samples = [metric(true_values[index], predicted_values[index]) for index in RESAMPLE_INDEX]
    low, high = np.percentile(samples, [2.5, 97.5])
    return f"{low:.2f}-{high:.2f}"

def f2(true_values, predicted_values):
    tp = np.sum(true_values & predicted_values)
    return 5 * tp / max(5 * tp + 4 * np.sum(true_values & ~predicted_values) + np.sum(~true_values & predicted_values), 1)

def score_question(question, true_values, predicted_values, probabilities=None):
    question_type = QUESTIONS[question]["type"]
    if question_type == "choice":
        return {"accuracy": accuracy_score(true_values, predicted_values),
                "95% CI": bootstrap_ci(accuracy_score, true_values, predicted_values),
                "macro F1": f1_score(true_values, predicted_values, average="macro", zero_division=0)}
    if question_type == "noul":
        true_values, predicted_values = np.asarray(true_values, bool), np.asarray(predicted_values, bool)
        tp = int(np.sum(true_values & predicted_values))
        result = {"F2": f2(true_values, predicted_values), "95% CI": bootstrap_ci(f2, true_values, predicted_values),
                  "recall": tp / max(true_values.sum(), 1), "precision": tp / max(predicted_values.sum(), 1),
                  "TP": tp, "FN": int(np.sum(true_values & ~predicted_values)), "FP": int(np.sum(~true_values & predicted_values))}
        if probabilities is not None:
            result["AUC"] = roc_auc_score(true_values, probabilities)
        return result
    quadratic_kappa = lambda a, b: cohen_kappa_score(a, b, weights="quadratic", labels=list(range(len(RESOLUTION_LEVELS))))
    return {"accuracy": accuracy_score(true_values, predicted_values),
            "95% CI": bootstrap_ci(accuracy_score, true_values, predicted_values),
            "weighted kappa": quadratic_kappa(true_values, predicted_values)}

all_predictions, all_probabilities = {}, {}   # model -> question -> array over test rows

def score_model(model_name):
    return {question: score_question(question, test[question].values, all_predictions[model_name][question],
                                     all_probabilities.get(model_name, {}).get(question))
            for question in QUESTIONS}

## 4. Keyword baseline

The zero-training floor: one regular expression per yes/no label, written from the label names. `01_explore.ipynb` showed these keywords already separate the labels well, so any model has to beat `grep` to earn credit. There is no keyword version of the two choice questions (55 intents don't reduce to one regex each), so it's scored on the yes/no and resolution questions only.

In [ ]:
KEYWORDS = {
    "identity_verified": r"verif|zip ?code|account id",
    "refund_offered": r"\brefund",
    "promo_given": r"\bpromo|\bcode\b|discount",
    "handed_off": r"manager|supervisor|department|team|escalat",
}
CHANGE_KEYWORDS = r"\bupdat|\bchang|\bcancel|\breset|new password|\border(?:ed)? (?:it|that|one)"

def keyword_predictions(frame):
    predictions = {question: frame["text"].str.contains(pattern, flags=re.IGNORECASE, regex=True).values
                   for question, pattern in KEYWORDS.items()}
    money = predictions["refund_offered"] | predictions["promo_given"]
    changed = frame["text"].str.contains(CHANGE_KEYWORDS, flags=re.IGNORECASE, regex=True).values
    predictions["resolution"] = np.where(money, 2, np.where(changed, 1, 0))
    return predictions

all_predictions["Keywords"] = keyword_predictions(test)
keyword_scores = {question: score_question(question, test[question].values, prediction)
                  for question, prediction in all_predictions["Keywords"].items()}
pd.DataFrame(keyword_scores).T

## 5. Supervised baseline and learning curve

TF-IDF (words and word pairs) + logistic regression, one model per question, trained on the train split. Neutral settings (no class weighting, 0.5 threshold), like the neutral RF in the attrition run.

The **learning curve** answers the practical question behind "just train a model": how many hand-labeled conversations does it take to match each zero-shot model? Each size is drawn 5 times at random; the curve shows the mean.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

def train_tfidf(training_rows, question):
    model = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
                          LogisticRegression(max_iter=2000, C=4.0))
    return model.fit(training_rows["text"], training_rows[question])

def headline_metric(question, true_values, predicted_values):
    kind = QUESTIONS[question]["type"]
    if kind == "noul":
        return f2(np.asarray(true_values, bool), np.asarray(predicted_values, bool))
    return accuracy_score(true_values, predicted_values)

SUPERVISED = f"TF-IDF + LR ({len(train):,} labels)"
all_predictions[SUPERVISED], all_probabilities[SUPERVISED] = {}, {}
for question in QUESTIONS:
    model = train_tfidf(train, question)
    all_predictions[SUPERVISED][question] = model.predict(test["text"])
    if QUESTIONS[question]["type"] == "noul":
        all_probabilities[SUPERVISED][question] = model.predict_proba(test["text"])[:, 1]
pd.DataFrame(score_model(SUPERVISED)).T

In [ ]:
import warnings

LEARNING_CURVE_SIZES = [25, 50, 100, 200, 500, 1000, 2000, 4000, len(train)]
warnings.filterwarnings("ignore", message="The number of unique classes")   # tiny draws of the 55-way intent question
LEARNING_CURVE_DRAWS = 5
curve_rng = np.random.default_rng(SEED)
learning_curve_rows = []
for size in LEARNING_CURVE_SIZES:
    for draw in range(LEARNING_CURVE_DRAWS if size < len(train) else 1):
        sample = train.iloc[curve_rng.choice(len(train), size=size, replace=False)]
        for question in QUESTIONS:
            if sample[question].nunique() < 2:   # tiny draws can miss a rare class entirely: count as a failed model
                learning_curve_rows.append({"size": size, "draw": draw, "question": question, "score": 0.0})
                continue
            predicted = train_tfidf(sample, question).predict(test["text"])
            learning_curve_rows.append({"size": size, "draw": draw, "question": question,
                                        "score": headline_metric(question, test[question].values, predicted)})
learning_curve = pd.DataFrame(learning_curve_rows).groupby(["question", "size"])["score"].mean().unstack("size")
learning_curve.round(3)

## 6. One prompt, three request shapes

Everything is rendered from `QUESTIONS`:
- **Jev / Laya**: the transcript is `state`; each question is a typed `choice` / `noul` / `score` question with the same instructions and option descriptions.
- **LLMs**: the same instructions and descriptions as text, the transcript last (so the identical question block can be cached), and a JSON schema that only allows the listed options.

For score answers, Jev and Laya return a distribution over levels. Their docs warn that the expected value is unreliable as a magnitude, so the **most likely level** is used.

In [ ]:
LLM_TASK = ("You are labeling a customer-service conversation between an agent and a customer of an online clothing store. "
            "Answer every question using only the conversation. Reply with a JSON object that has one key per question, "
            "using exactly one of the listed option keys as each value.")

def render_question_block():
    lines = []
    for name, question in QUESTIONS.items():
        lines.append(f"### {name}\n{question['instructions']}")
        if question["type"] == "noul":
            lines.append(f"- yes: {question['criteria']['true']}\n- no: {question['criteria']['false']}")
        else:
            lines.extend(f"- {key}: {description}" for key, description in question["criteria"].items())
        lines.append("")
    return "\n".join(lines)

LLM_QUESTION_BLOCK = f"{LLM_TASK}\n\n## Questions\n\n{render_question_block()}"

def llm_option_keys(name):
    question = QUESTIONS[name]
    return ["yes", "no"] if question["type"] == "noul" else list(question["criteria"])

LLM_SCHEMA = {
    "type": "object",
    "properties": {name: {"type": "string", "enum": llm_option_keys(name)} for name in QUESTIONS},
    "required": list(QUESTIONS),
    "additionalProperties": False,
}

def llm_prompt_parts(transcript):
    return LLM_QUESTION_BLOCK, f"## Conversation\n{transcript}\n"   # (identical cacheable prefix, per-call suffix)

def parse_llm_answers(raw_text):
    # Every answer checked against its option list; anything else -> None (invalid, scored as wrong)
    try:
        answers = json.loads(raw_text)
    except (json.JSONDecodeError, TypeError):
        return {name: None for name in QUESTIONS}
    parsed = {}
    for name, question in QUESTIONS.items():
        value = answers.get(name) if isinstance(answers, dict) else None
        if value not in llm_option_keys(name):
            parsed[name] = None
        elif question["type"] == "noul":
            parsed[name] = value == "yes"
        elif question["type"] == "score":
            parsed[name] = list(question["criteria"]).index(value)
        else:
            parsed[name] = value
    return parsed

def system_one_questions():
    # Plain JSON in Jev's wire format; Laya's Jev-compatible API takes the same shape
    rendered = {}
    for name, question in QUESTIONS.items():
        if question["type"] == "score":
            rendered[name] = {"type": "score", "instructions": question["instructions"], "criteria": list(question["criteria"].values())}
        else:
            rendered[name] = {"type": question["type"], "instructions": question["instructions"], "criteria": question["criteria"]}
    return rendered

SYSTEM_ONE_QUESTIONS = system_one_questions()

def parse_system_one_answers(answers):
    # answers: name -> {"choice"/"noul"/"score", "probabilities"}; returns (labels, probabilities for yes/no)
    labels, probabilities = {}, {}
    for name, question in QUESTIONS.items():
        answer = answers[name]
        if question["type"] == "choice":
            labels[name] = answer["choice"]
        elif question["type"] == "noul":
            probabilities[name] = float(answer["noul"])
            labels[name] = probabilities[name] >= 0.5
        else:
            level_probabilities = {int(level): p for level, p in answer["probabilities"].items()}
            labels[name] = max(level_probabilities, key=level_probabilities.get)
    return labels, probabilities

print(LLM_QUESTION_BLOCK[:1500], "...")
print(f"\nquestion block: {len(LLM_QUESTION_BLOCK):,} chars")
print("parser check:", parse_llm_answers('{"reason": "shipping_issue"}')["reason"], parse_llm_answers("not json")["reason"])

## 7. The models

| Model | Tier | Reasoning (provider default) | Structured output |
|---|---|---|---|
| Claude Sonnet 5 | mid | adaptive thinking, effort high | `output_config.format` json_schema |
| GPT-6 Sol | mid | effort medium | `text.format` json_schema, strict |
| Gemini 3.8 Flash | mid | thinking level medium | `response_json_schema` |
| Claude Haiku 4.5 | cheap | none (no thinking by default) | `output_config.format` json_schema |
| Jev 1.13 | System One API | n/a (calibrated probabilities) | typed answers |
| Laya (English, 512) | System One, local | n/a | typed answers |
| Laya (English, 1,024) | System One, local | n/a | typed answers, longer budget |

- **Prices** are $ per 1M tokens, the same table as the attrition run, plus Haiku 4.5 ($1 input / $0.10 cache read / $1.25 cache write / $5 output).
- **Caching**: the ~2k-token question block is identical on every call. Sonnet 5 and GPT-6 get an explicit breakpoint before the transcript. Haiku 4.5's minimum cacheable prompt is 4,096 tokens, so it can't cache this prefix; that's a real cost of the cheap tier here, not a setup mistake.
- **Laya runs twice.** The English checkpoint's default budget is 512 tokens, about 320 of them for the transcript, so the longest ~15% of conversations are cut. The second run gives it 1,024 tokens (its documented override). The option list shares a separate budget, which the 55-option intent question will strain; Laya's own docs list many-option questions as a weakness.
- **Laya's cost** is $0 per call on your own GPU; hardware and power aren't counted, same as the local model in the attrition run.
- **No tools or web search** for any model; OpenAI calls use `store=False`.

In [ ]:
import csv
from dataclasses import asdict, dataclass, fields
from datetime import datetime, timezone

import anthropic
import openai
from google import genai
from google.genai import types as genai_types
from typesafe_sdk import TypeSafeClient

JEV_MODEL = "jev-1.13.0"   # pinned, not jev-latest
MAX_OUTPUT_TOKENS = 16_000

MODELS = {
    "Claude Sonnet 5":  {"model_id": "claude-sonnet-5", "key_env": "CLAUDE_KEY", "reasoning": "adaptive thinking, effort high",
                         "price": {"input": 2.00, "cached_input": 0.20, "cache_write": 2.50, "output": 10.00}},
    "GPT-6 Sol":        {"model_id": "gpt-6-sol", "key_env": "GPT_KEY", "reasoning": "effort medium",
                         "price": {"input": 2.00, "cached_input": 0.20, "cache_write": 2.50, "output": 10.00}},
    "Gemini 3.8 Flash": {"model_id": "gemini-3.8-flash", "key_env": "GEM_KEY", "reasoning": "thinking_level medium",
                         "price": {"input": 0.75, "cached_input": 0.075, "cache_write": 0.00, "output": 3.75}},
    "Claude Haiku 4.5": {"model_id": "claude-haiku-4-5", "key_env": "CLAUDE_KEY", "reasoning": "none (default)",
                         "price": {"input": 1.00, "cached_input": 0.10, "cache_write": 1.25, "output": 5.00}},
    "Jev":              {"model_id": JEV_MODEL, "key_env": "JEV_KEY", "reasoning": "n/a",
                         "price": {"input": 0.042, "cached_input": 0.042, "cache_write": 0.00, "output": 0.00}},
    "Laya (512)":       {"model_id": "convaiinnovations/laya", "key_env": None, "reasoning": "n/a", "max_len": None,
                         "price": {"input": 0.0, "cached_input": 0.0, "cache_write": 0.0, "output": 0.0}},
    "Laya (1,024)":     {"model_id": "convaiinnovations/laya", "key_env": None, "reasoning": "n/a", "max_len": 1024,
                         "price": {"input": 0.0, "cached_input": 0.0, "cache_write": 0.0, "output": 0.0}},
}
# Laya needs `pip install laya` and downloads ~1.7 GB of weights from Hugging Face on first load
INCLUDE_LAYA = False
if not INCLUDE_LAYA:
    for model_name in ["Laya (512)", "Laya (1,024)"]:
        MODELS.pop(model_name)

@dataclass
class CallRecord:
    model: str
    convo_id: int
    variant: str                           # "original" or "perturbed" (memorization probe)
    raw_answer: str = ""
    answers: str = ""                      # JSON: question -> parsed label (None = invalid)
    probabilities: str = ""                # JSON: yes/no question -> P(yes), System One models only
    invalid_answers: int = 0
    latency_s: float = 0.0
    uncached_input_tokens: int = 0
    cached_input_tokens: int = 0
    cache_write_tokens: int = 0
    output_tokens: int = 0                 # includes reasoning tokens (billed as output)
    reasoning_tokens: int = 0
    cost_usd: float = 0.0
    served_model: str = ""
    error: str = ""
    timestamp: str = ""

def api_key(env_name, required=True):
    key = os.environ.get(env_name)
    if not key and os.name == "nt":
        # Set in Windows AFTER Jupyter started: read the user environment directly
        import winreg
        try:
            with winreg.OpenKey(winreg.HKEY_CURRENT_USER, "Environment") as user_environment:
                key = winreg.QueryValueEx(user_environment, env_name)[0]
        except FileNotFoundError:
            key = None
    assert key or not required, f"set {env_name}"
    return key

def call_claude(model_name, transcript, client):
    prefix, suffix = llm_prompt_parts(transcript)
    thinking = {"thinking": {"type": "adaptive"}, "output_config": {"effort": "high"}} if model_name == "Claude Sonnet 5" else {"output_config": {}}
    thinking["output_config"]["format"] = {"type": "json_schema", "schema": LLM_SCHEMA}
    response = client.messages.create(
        model=MODELS[model_name]["model_id"],
        max_tokens=MAX_OUTPUT_TOKENS,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prefix, "cache_control": {"type": "ephemeral"}},
            {"type": "text", "text": suffix},
        ]}],
        **thinking,
    )
    if response.stop_reason == "refusal":
        raise RuntimeError(f"refusal: {response.stop_details}")
    usage = response.usage
    return {
        "raw_answer": "".join(block.text for block in response.content if block.type == "text"),
        "uncached_input_tokens": usage.input_tokens,
        "cached_input_tokens": usage.cache_read_input_tokens or 0,
        "cache_write_tokens": usage.cache_creation_input_tokens or 0,
        "output_tokens": usage.output_tokens,
        "served_model": response.model,
    }

def call_gpt(model_name, transcript, client):
    prefix, suffix = llm_prompt_parts(transcript)
    response = client.responses.create(
        model=MODELS[model_name]["model_id"],
        input=[{"role": "user", "content": [
            {"type": "input_text", "text": prefix, "prompt_cache_breakpoint": {"mode": "explicit"}},
            {"type": "input_text", "text": suffix},
        ]}],
        reasoning={"effort": "medium"},
        text={"format": {"type": "json_schema", "name": "conversation_labels", "schema": LLM_SCHEMA, "strict": True}},
        max_output_tokens=MAX_OUTPUT_TOKENS,
        prompt_cache_key="abcd-call-labeling",
        store=False,
    )
    usage = response.usage
    cached = usage.input_tokens_details.cached_tokens or 0
    cache_written = getattr(usage.input_tokens_details, "cache_write_tokens", 0) or 0
    return {
        "raw_answer": response.output_text,
        "uncached_input_tokens": usage.input_tokens - cached - cache_written,
        "cached_input_tokens": cached,
        "cache_write_tokens": cache_written,
        "output_tokens": usage.output_tokens,
        "reasoning_tokens": usage.output_tokens_details.reasoning_tokens or 0,
        "served_model": response.model,
    }

def call_gemini(model_name, transcript, client):
    prefix, suffix = llm_prompt_parts(transcript)
    response = client.models.generate_content(
        model=MODELS[model_name]["model_id"],
        contents=prefix + "\n" + suffix,
        config=genai_types.GenerateContentConfig(
            thinking_config=genai_types.ThinkingConfig(thinking_level="medium"),
            response_mime_type="application/json",
            response_json_schema=LLM_SCHEMA,
            max_output_tokens=MAX_OUTPUT_TOKENS,
        ),
    )
    usage = response.usage_metadata
    cached = usage.cached_content_token_count or 0
    thoughts = usage.thoughts_token_count or 0
    return {
        "raw_answer": response.text or "",
        "uncached_input_tokens": usage.prompt_token_count - cached,
        "cached_input_tokens": cached,
        "output_tokens": (usage.candidates_token_count or 0) + thoughts,
        "reasoning_tokens": thoughts,
        "served_model": response.model_version or "",
    }

def call_jev(model_name, transcript, client):
    response = client.system_one(state=transcript, questions=SYSTEM_ONE_QUESTIONS, model=JEV_MODEL)
    answers = {name: answer.model_dump() for name, answer in response.answers.items()}
    return {
        "raw_answer": json.dumps(answers),
        "system_one_answers": answers,
        "uncached_input_tokens": response.usage.input_tokens or 0,
        "output_tokens": response.usage.output_tokens or 0,
        "served_model": response.model,
    }

def call_laya(model_name, transcript, agent):
    budget = {"max_len": MODELS[model_name]["max_len"], "head_max_len": 512} if MODELS[model_name]["max_len"] else {}
    result = agent.predict(transcript, SYSTEM_ONE_QUESTIONS, **budget)
    answers = result["answers"]
    return {
        "raw_answer": json.dumps(answers, default=float),
        "system_one_answers": answers,
        "uncached_input_tokens": result.get("usage", {}).get("input_tokens", 0),
        "served_model": MODELS[model_name]["model_id"],
    }

MODEL_CALLERS = {"Claude Sonnet 5": call_claude, "Claude Haiku 4.5": call_claude, "GPT-6 Sol": call_gpt,
                 "Gemini 3.8 Flash": call_gemini, "Jev": call_jev, "Laya (512)": call_laya, "Laya (1,024)": call_laya}

_laya_agent = None
def make_client(model_name):
    global _laya_agent
    if model_name.startswith("Laya"):
        if _laya_agent is None:
            from laya import load
            _laya_agent = load(MODELS[model_name]["model_id"])   # load BEFORE timing starts
        return _laya_agent
    key = api_key(MODELS[model_name]["key_env"])
    if model_name.startswith("Claude"):
        workspace_id = api_key("CLAUDE_WORKSPACE_ID", required=False)
        return anthropic.Anthropic(api_key=key, default_headers={"anthropic-workspace-id": workspace_id} if workspace_id else None)
    return {"GPT-6 Sol": lambda: openai.OpenAI(api_key=key),
            "Gemini 3.8 Flash": lambda: genai.Client(api_key=key),
            "Jev": lambda: TypeSafeClient(api_key=key, model=JEV_MODEL, timeout=60)}[model_name]()

def cost_usd(model_name, usage):
    price = MODELS[model_name]["price"]
    return (usage.get("uncached_input_tokens", 0) * price["input"] + usage.get("cached_input_tokens", 0) * price["cached_input"]
            + usage.get("cache_write_tokens", 0) * price["cache_write"] + usage.get("output_tokens", 0) * price["output"]) / 1_000_000

def label_one(model_name, convo_id, transcript, client, variant="original"):
    # One conversation per call. Latency = wall clock around the API call only.
    record = CallRecord(model=model_name, convo_id=int(convo_id), variant=variant, timestamp=datetime.now(timezone.utc).isoformat())
    started = time.perf_counter()
    try:
        result = MODEL_CALLERS[model_name](model_name, transcript, client)
    except Exception as error:   # logged, never silently dropped; the run loop retries failures
        record.latency_s = time.perf_counter() - started
        record.error = f"{type(error).__name__}: {error}"[:500]
        return record
    record.latency_s = time.perf_counter() - started
    system_one_answers = result.pop("system_one_answers", None)
    for field_name, value in result.items():
        setattr(record, field_name, value)
    if system_one_answers is not None:
        labels, probabilities = parse_system_one_answers(system_one_answers)
        record.probabilities = json.dumps(probabilities)
    else:
        labels = parse_llm_answers(record.raw_answer)
    record.answers = json.dumps(labels, default=lambda value: bool(value) if isinstance(value, np.bool_) else int(value))
    record.invalid_answers = sum(value is None for value in labels.values())
    record.cost_usd = cost_usd(model_name, result)
    return record

## 8. Smoke test

One call per model on one **dev** conversation, to prove the wiring before spending on the full run.

In [ ]:
RUN_SMOKE_TEST = False
if RUN_SMOKE_TEST:
    smoke_row = dev.iloc[0]
    smoke_records = [label_one(model_name, smoke_row["convo_id"], smoke_row["text"], make_client(model_name)) for model_name in MODELS]
    smoke_table = pd.DataFrame([asdict(record) for record in smoke_records]).set_index("model")
    display(smoke_table[["served_model", "answers", "invalid_answers", "latency_s", "uncached_input_tokens",
                         "cached_input_tokens", "cache_write_tokens", "output_tokens", "cost_usd", "error"]])
    print("truth:", {name: smoke_row[name] for name in QUESTIONS})

## 9. The run

1,000 test conversations x every model, one conversation per call, sequential, same order for every model. Each call is appended to `call_log.csv` the moment it returns, so the run survives crashes and resumes where it stopped. The log holds conversation IDs, answers, probabilities, tokens, latency and cost; never transcript text.

In [ ]:
CALL_LOG_PATH = Path("call_log.csv")
RUN_MODELS = list(MODELS)   # e.g. ["Jev"] to run one model at a time
MAX_ATTEMPTS = 3
LOG_COLUMNS = [field.name for field in fields(CallRecord)]
NUMERIC_COLUMNS = ["convo_id", "invalid_answers", "latency_s", "uncached_input_tokens", "cached_input_tokens",
                   "cache_write_tokens", "output_tokens", "reasoning_tokens", "cost_usd"]

def append_to_log(record):
    is_new_log = not CALL_LOG_PATH.exists()
    with CALL_LOG_PATH.open("a", newline="", encoding="utf-8") as log_file:
        writer = csv.DictWriter(log_file, fieldnames=LOG_COLUMNS)
        if is_new_log:
            writer.writeheader()
        writer.writerow(asdict(record))

def load_call_log():
    if not CALL_LOG_PATH.exists():
        return pd.DataFrame(columns=LOG_COLUMNS)
    call_log = pd.read_csv(CALL_LOG_PATH, dtype=str, keep_default_na=False)
    for column in NUMERIC_COLUMNS:
        call_log[column] = pd.to_numeric(call_log[column], errors="coerce")
    return call_log

def run(model_names, frame, variant="original", text_for=lambda row: row["text"]):
    for model_name in model_names:
        call_log = load_call_log()
        done = set(call_log.loc[(call_log["model"] == model_name) & (call_log["variant"] == variant) & (call_log["error"] == ""), "convo_id"])
        remaining = [row for _, row in frame.iterrows() if row["convo_id"] not in done]
        print(f"{model_name} [{variant}]: {len(frame) - len(remaining)} already done, {len(remaining)} to run")
        if not remaining:
            continue
        client = make_client(model_name)
        for attempt in range(1, MAX_ATTEMPTS + 1):
            failed, last_error = [], ""
            for position, row in enumerate(remaining, start=1):
                record = label_one(model_name, row["convo_id"], text_for(row), client, variant)
                append_to_log(record)
                if record.error:
                    failed.append(row)
                    last_error = record.error
                if position % 100 == 0:
                    print(f"  {position}/{len(remaining)} calls, {len(failed)} failed so far")
            if not failed:
                break
            print(f"  attempt {attempt}: {len(failed)} failed, last error: {last_error[:160]}")
            remaining = failed
            time.sleep(15 * attempt)

RUN_BENCHMARK = False
if RUN_BENCHMARK:
    run(RUN_MODELS, test)

## 10. Results

Every model with an answer for all 1,000 test conversations, scored with the shared scorers. Invalid answers count as wrong: for yes/no questions they count as "no", so an unparseable answer can never count as a catch.

In [ ]:
def collect(variant="original", frame=test):
    call_log = load_call_log()
    successful = call_log[(call_log["error"] == "") & (call_log["variant"] == variant)].drop_duplicates(["model", "convo_id"])
    collected = {}
    for model_name in MODELS:
        calls = successful[successful["model"] == model_name].set_index("convo_id")
        if not set(frame["convo_id"]) <= set(calls.index):
            print(f"{model_name} [{variant}]: {len(calls)}/{len(frame)} answered, not scored yet")
            continue
        calls = calls.loc[frame["convo_id"]]
        answers = [json.loads(text) for text in calls["answers"]]
        probabilities = [json.loads(text) if text else {} for text in calls["probabilities"]]
        predictions = {}
        for question, spec in QUESTIONS.items():
            values = [answer.get(question) for answer in answers]
            if spec["type"] == "noul":
                predictions[question] = np.array([bool(value) for value in values])   # None -> False
            elif spec["type"] == "score":
                predictions[question] = np.array([-1 if value is None else value for value in values])
            else:
                predictions[question] = np.array(["<invalid>" if value is None else value for value in values], dtype=object)
        model_probabilities = {question: np.array([p[question] for p in probabilities])
                               for question in NOUL_QUESTIONS if probabilities and question in probabilities[0]}
        collected[model_name] = (predictions, model_probabilities, calls)
    return collected

efficiency = {}
for model_name, (predictions, model_probabilities, calls) in collect().items():
    all_predictions[model_name] = predictions
    if model_probabilities:
        all_probabilities[model_name] = model_probabilities
    calls = calls.assign(cost_usd=[cost_usd(model_name, row) for row in calls.to_dict("records")])   # current price table
    total_input = calls[["uncached_input_tokens", "cached_input_tokens", "cache_write_tokens"]].sum(axis=1)
    efficiency[model_name] = {
        "invalid answers": int(calls["invalid_answers"].sum()),
        "median latency s": calls["latency_s"].median(),
        "p90 latency s": calls["latency_s"].quantile(0.9),
        "cost per 1,000 conversations $": calls["cost_usd"].sum() / len(calls) * 1000,
        "input tokens / call": total_input.mean(),
        "cache hit share": calls["cached_input_tokens"].sum() / max(total_input.sum(), 1),
        "output tokens / call": calls["output_tokens"].mean(),
        "reasoning tokens / call": calls["reasoning_tokens"].mean(),
        "served model": ", ".join(sorted(calls["served_model"].unique())),
    }

scores = {model_name: score_model(model_name) for model_name in all_predictions if model_name != "Keywords"}
scores["Keywords"] = {question: score_question(question, test[question].values, prediction)
                      for question, prediction in all_predictions["Keywords"].items()}

def question_table(question):
    return pd.DataFrame({model_name: model_scores[question] for model_name, model_scores in scores.items()
                         if question in model_scores}).T

for question in QUESTIONS:
    print(f"\n=== {question} ({QUESTIONS[question]['type']}) ===")
    display(question_table(question))
efficiency_table = pd.DataFrame(efficiency).T
display(efficiency_table)

### Labels needed to match each zero-shot model

For each question, the smallest training-set size at which the supervised baseline's average score reaches the zero-shot model's score. "> 8,006" means even the full training set (all 8,006 usable train conversations) didn't get there.

In [ ]:
def labels_to_match(question, target):
    reached = learning_curve.loc[question][learning_curve.loc[question] >= target]
    return f"{reached.index[0]:,}" if len(reached) else f"> {len(train):,}"

headline_names = {"choice": "accuracy", "noul": "F2", "score": "accuracy"}
zero_shot_models = [model_name for model_name in scores if model_name in MODELS]
labels_needed = pd.DataFrame({
    model_name: {question: labels_to_match(question, scores[model_name][question][headline_names[QUESTIONS[question]["type"]]])
                 for question in QUESTIONS}
    for model_name in zero_shot_models
})
labels_needed

## 11. Memorization probe

ABCD has been public since 2021. If a model has memorized it, it could label a conversation by recognizing it rather than reading it. The probe swaps every identifying detail (names, usernames, emails, phone numbers, order and account IDs, addresses, zip codes) for fresh fake values on 200 test conversations and runs them again. A model that reads should score about the same; a model that recognizes rows should drop.

The swap is an exact-string match against the scenario, so a customer's own typo of their name ("Rodriquez" for "rodriguez") slips through; the IDs, which carry the recognizable detail, are always swapped.

In [ ]:
PROBE_SIZE = 200
probe_rows = test.sample(n=PROBE_SIZE, random_state=SEED).sort_values("convo_id")
fake_rng = random.Random(SEED)
FAKE_FIRST = ["jordan", "riley", "casey", "morgan", "avery", "quinn", "rowan", "emerson", "sage", "harper"]
FAKE_LAST = ["okafor", "lindqvist", "moreau", "tanaka", "castillo", "novak", "brennan", "haddad", "ivanova", "mensah"]

def fake_like(value):
    # Same shape, new content: digits -> random digits, letters -> random letters
    return "".join(str(fake_rng.randint(0, 9)) if ch.isdigit() else fake_rng.choice("abcdefghijklmnopqrstuvwxyz") if ch.isalpha() else ch
                   for ch in value)

def perturb(convo_id):
    scenario = conversation_by_id[convo_id]["scenario"]
    replacements = {}
    first, *rest = scenario["personal"]["customer_name"].split()
    new_first, new_last = fake_rng.choice(FAKE_FIRST), fake_rng.choice(FAKE_LAST)
    replacements[scenario["personal"]["customer_name"]] = f"{new_first} {new_last}"
    replacements[first] = new_first
    for last in rest:
        replacements[last] = new_last
    for field in ["email", "phone", "username", "account_id"]:
        if scenario["personal"].get(field):
            replacements[scenario["personal"][field]] = fake_like(scenario["personal"][field])
    for field in ["order_id", "zip_code", "street_address", "full_address"]:
        if scenario["order"].get(field):
            replacements[scenario["order"][field]] = fake_like(scenario["order"][field])
    text = test.loc[test["convo_id"] == convo_id, "text"].iloc[0]
    for original in sorted(replacements, key=len, reverse=True):   # longest first, so full names go before first names
        text = re.sub(re.escape(original), replacements[original], text, flags=re.IGNORECASE)
    return text

perturbed_text = {convo_id: perturb(convo_id) for convo_id in probe_rows["convo_id"]}
example_id = next(convo_id for convo_id in probe_rows["convo_id"]   # show one where something was actually swapped
                  if perturbed_text[convo_id] != test.loc[test["convo_id"] == convo_id, "text"].iloc[0])
print(test.loc[test["convo_id"] == example_id, "text"].iloc[0][:600], "\n---\n", perturbed_text[example_id][:600])

RUN_MEMORIZATION_PROBE = False
if RUN_MEMORIZATION_PROBE:
    run(list(MODELS), probe_rows, variant="perturbed", text_for=lambda row: perturbed_text[row["convo_id"]])

probe_results = {}
perturbed = collect("perturbed", probe_rows)
for model_name, (predictions, _, _) in perturbed.items():
    probe_positions = test.index[test["convo_id"].isin(probe_rows["convo_id"])]
    original_accuracy = np.mean([np.mean(all_predictions[model_name][question][probe_positions] == test[question].values[probe_positions])
                                 for question in QUESTIONS])
    perturbed_accuracy = np.mean([np.mean(predictions[question] == probe_rows[question].values) for question in QUESTIONS])
    probe_results[model_name] = {"mean accuracy, original": original_accuracy, "mean accuracy, details swapped": perturbed_accuracy,
                                 "change": perturbed_accuracy - original_accuracy}
pd.DataFrame(probe_results).T

## 12. Cost at scale

The measured cost per conversation, projected to a support team labeling **10 million conversations a month**. Latency is the median single call measured here; parallel throughput is a separate experiment.

In [ ]:
MONTHLY_CONVERSATIONS = 10_000_000
if len(efficiency_table):
    at_scale = efficiency_table[["cost per 1,000 conversations $", "median latency s"]].copy()
    at_scale["cost per month at 10M $"] = at_scale["cost per 1,000 conversations $"] * MONTHLY_CONVERSATIONS / 1000
    display(at_scale.sort_values("cost per month at 10M $"))

## Caveats

To be filled in from the results. Known in advance:
- **Role-played data.** ABCD conversations are crowdworkers following scripts, typed in chat. The commercially usable real call corpora don't exist; the principle being tested is labeling conversations, not the realness of the set.
- **Keyword regexes were written after looking at the whole dataset** in `01_explore.ipynb`, so the keyword baseline is, if anything, slightly flattered.
- **Option descriptions are one honest attempt**, identical for every model. Better descriptions would likely help every zero-shot model; they were not tuned.
- **The cloud LLMs aren't deterministic**; the attrition run saw F2 move by 0.07 between identical runs. Jev returned identical probabilities on every repeat.
- **Token counts aren't comparable across vendors.** Compare cost and latency.